In [53]:
import pandas as pd 
import os 
from sqlalchemy import create_engine
!pip install pymysql

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
username = "root"
password = "YOUR_MYSQL_PASSWORD"
host = "localhost"
port = 3306
database = "vendor_db"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)


Connected successfully!


In [ ]:
four_tables = [
    'begin_inventory.csv',
    'end_inventory.csv',
    'purchase_prices.csv',
    'purchases.csv'
]

for file in os.listdir(r'C:\Users\ffrah\data'):
    if file in four_tables:
        df = pd.read_csv(os.path.join(r'C:\Users\ffrah\data', file))
        ingest_db(df, file[:-4], engine)


def ingest_db(df, table_name, engine):
    df.to_sql(
        table_name,
        con=engine,
        if_exists='replace',
        index=False
    )

In [19]:
def ingest_sales(file_path, table_name, engine):
    first_chunk = True

    for chunk in pd.read_csv(file_path, chunksize=50000):
        chunk.to_sql(
            table_name,
            con=engine,
            if_exists='replace' if first_chunk else 'append',
            index=False,
            method='multi'
        )

        print(f"Inserted {len(chunk):,} rows")
        first_chunk = False

    print("Sales ingestion completed!")
    
sales_file = r'C:\Users\ffrah\data\sales.csv'

ingest_sales(sales_file, 'sales', engine)

In [ ]:
vendor_invoice = pd.read_csv(r'C:\Users\ffrah\data\vendor_invoice.csv')
vendor_invoice.to_sql(
    'vendor_invoice',
    con=engine,
    if_exists='replace',
    index=False
)